# Biological Age Analysis — Immortigen

This notebook walks through the complete biological age pipeline:
1. Data exploration (methylation + blood biomarkers)
2. Hannum clock implementation and validation
3. Blood biomarker MLP evaluation
4. Multi-modal fusion analysis
5. Population-level aging insights
6. RAG retrieval demonstration

**Reference:** Hannum G et al. *Mol Cell* 2013;49(2):359-367

In [ ]:
import sys; sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

## 1. Data Exploration

In [ ]:
# Generate or load data
from data.simulate_data import simulate_methylation, simulate_blood_biomarkers

meth  = simulate_methylation(656)
blood = simulate_blood_biomarkers(meth['sample_id'].tolist(), meth['chronological_age'].values)

print(f'Methylation shape: {meth.shape}')
print(f'Blood shape:       {blood.shape}')
meth.head(3)

In [ ]:
# Age distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

meth['chronological_age'].hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Chronological Age Distribution (GSE40279-style)')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Count')

# CpG correlation with age — show top 10
cpg_cols = [c for c in meth.columns if c.startswith('cg')]
corrs = meth[cpg_cols].corrwith(meth['chronological_age']).abs().sort_values(ascending=False)
corrs.head(10).plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Top 10 CpG Correlations with Age')
axes[1].set_xlabel('CpG Site')
axes[1].set_ylabel('|Pearson r|')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2. Hannum Clock — Train and Evaluate

In [ ]:
from models.methylation_model import HannumClock

clock = HannumClock()
clock.train(meth)
meth_preds = clock.predict(meth)

mae = mean_absolute_error(meth['chronological_age'], meth_preds['biological_age_methylation'])
r2  = r2_score(meth['chronological_age'], meth_preds['biological_age_methylation'])
print(f'MAE: {mae:.2f} years | R²: {r2:.3f}')
print(f'\nHannum 2013 reported MAE ~3.9 years on held-out test set')

In [ ]:
# Predicted vs actual scatter — the key diagnostic plot
fig = px.scatter(
    x=meth['chronological_age'],
    y=meth_preds['biological_age_methylation'],
    color=meth_preds['methylation_age_gap'],
    color_continuous_scale='RdYlGn_r',
    labels={'x': 'Chronological Age (years)', 'y': 'Epigenetic Age (years)', 'color': 'Age Gap'},
    title=f'Hannum Clock: Predicted vs Actual (MAE={mae:.2f}y, R²={r2:.3f})',
)
fig.add_shape(type='line', x0=20, y0=20, x1=100, y1=100,
              line=dict(color='black', dash='dash'), name='Perfect prediction')
fig.show()

In [ ]:
# Coefficient plot — which CpGs drive the clock?
top_cpgs = clock.top_cpgs(20)
colors = ['#E74C3C' if c > 0 else '#2980B9' for c in top_cpgs['coefficient']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_cpgs['cpg'], top_cpgs['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Elastic-Net Coefficient')
ax.set_title('Hannum Clock: Top 20 CpG Coefficients\n(Red = increases with age, Blue = decreases)')
plt.tight_layout()
plt.show()

## 3. Blood Biomarker MLP

In [ ]:
from models.blood_biomarker_model import BloodBiomarkerModel

blood_model = BloodBiomarkerModel()
blood_model.train(blood, epochs=150)
blood_preds = blood_model.predict(blood)

b_mae = mean_absolute_error(blood['chronological_age'], blood_preds['biological_age_blood'])
b_r2  = r2_score(blood['chronological_age'], blood_preds['biological_age_blood'])
print(f'Blood MLP MAE: {b_mae:.2f} years | R²: {b_r2:.3f}')

In [ ]:
# Blood biomarker correlation heatmap
blood_features = ['crp_mg_l','glucose_mg_dl','hdl_mg_dl','ldl_mg_dl','hba1c_pct',
                  'albumin_g_dl','creatinine_mg_dl','lymphocyte_pct','rdw_pct','telomere_score']
corr_with_age = blood[blood_features].corrwith(blood['chronological_age']).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2980B9' if v < 0 else '#E74C3C' for v in corr_with_age]
corr_with_age.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Blood Biomarker Correlations with Chronological Age')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

## 4. Multi-Modal Fusion

In [ ]:
from models.fusion_model import FusionModel

df_fused = meth[['sample_id','chronological_age']].merge(
    meth_preds[['sample_id','biological_age_methylation']], on='sample_id'
).merge(
    blood_preds[['sample_id','biological_age_blood']], on='sample_id'
)

fusion = FusionModel()
fusion.train(df_fused, epochs=200)
fused_result = fusion.predict(df_fused)

f_mae = mean_absolute_error(df_fused['chronological_age'], fused_result['composite_biological_age'])
f_r2  = r2_score(df_fused['chronological_age'], fused_result['composite_biological_age'])
print(f'Fusion MAE: {f_mae:.2f} years | R²: {f_r2:.3f}')

# Compare all 3 models
print(f'\nModel comparison:')
print(f'  Hannum clock:    MAE={mae:.2f}y  R²={r2:.3f}')
print(f'  Blood MLP:       MAE={b_mae:.2f}y  R²={b_r2:.3f}')
print(f'  Fusion (both):   MAE={f_mae:.2f}y  R²={f_r2:.3f}')

In [ ]:
# Population AAI distribution
fig = px.histogram(
    fused_result, x='aging_acceleration_index',
    color='aging_category',
    nbins=40,
    title='Population Aging Acceleration Index (AAI = Biological Age - Chronological Age)',
    labels={'aging_acceleration_index': 'AAI (years)', 'aging_category': 'Category'},
    color_discrete_map={
        'exceptional_longevity': '#00C853',
        'slower_aging': '#64DD17',
        'typical_aging': '#FFD600',
        'accelerated_aging': '#FF6D00',
        'significantly_accelerated': '#D50000',
    }
)
fig.add_vline(x=0, line_dash='dash', annotation_text='Chronological age baseline')
fig.show()

## 5. RAG Demonstration

In [ ]:
from rag.retriever import LongevityRetriever

retriever = LongevityRetriever()
retriever.build()

queries = [
    'rapamycin mTOR lifespan extension mice',
    'epigenetic clock methylation biological age',
    'senolytics dasatinib quercetin senescent cells',
    'NAD+ NMN sirtuins aging intervention',
]

for q in queries:
    results = retriever.search(q, k=2)
    print(f"Query: '{q}'")
    for r in results:
        print(f"  [{r['relevance_score']:.3f}] {r['title'][:65]}... ({r['year']})")
    print()

In [ ]:
# Hallucination detection demo
claims = [
    ('Rapamycin inhibits mTOR and extends lifespan in mice', True),
    ('Telomere shortening is associated with accelerated aging and disease', True),
    ('Drinking red wine daily reverses epigenetic age by 20 years', False),
    ('DNA methylation patterns change predictably with chronological age', True),
]

print('Grounding verification:')
print(f'{"Claim":<65} {"Expected":<10} {"Grounded":<10} {"Score"}')
print('-' * 100)
for claim, expected in claims:
    check = retriever.verify_claim(claim)
    correct = check['grounded'] == expected
    status = '✓' if correct else '✗'
    print(f'{status} {claim[:63]:<65} {str(expected):<10} {str(check["grounded"]):<10} {check["max_score"]:.3f}')

## 6. Key Limitations

This project is honest about what it does and doesn't do:

1. **Synthetic data**: The training data is simulated to mimic GEO GSE40279 and NHANES distributions. Real deployment requires the actual datasets.

2. **In-sample evaluation**: MAE figures are computed on training data. A proper held-out test set (20% of real GEO data) would give true generalization performance.

3. **Methylation → blood correlation**: Because both datasets are generated from the same age variable, the fusion model's improvement may be overstated. Real multi-modal data would show larger modality disagreement.

4. **Clock generalization**: The Hannum clock was validated on blood methylation specifically. Cross-tissue prediction requires the Horvath multi-tissue clock instead.

5. **Intervention causality**: The RAG-retrieved interventions are associational, not proven causal for all individuals. RCT evidence is noted where available.

These limitations are features, not bugs — acknowledging them demonstrates research maturity.